# Carga de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import random
import os

from torchvision import transforms
from sklearn.preprocessing import normalize
from PIL import Image
from tqdm import tqdm
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


# Carga de modelos

In [14]:
# Cargar el modelo DinoV2 desde torch hub
# https://github.com/facebookresearch/dinov2

#dino_v2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
dino_v2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
dino_v2 = dino_v2.to(device)
dino_v2.eval()

print("¡Modelo DinoV2 cargado exitosamente!")
print(f"Dimensión de salida del modelo: {dino_v2.embed_dim}")

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


¡Modelo DinoV2 cargado exitosamente!
Dimensión de salida del modelo: 384


In [15]:
!gdown --id 1-LqJ2bS_T8XOx5TClDq31MCkPPoXO4bZ
!unzip dinov3-main.zip


# DINOv3 ViT models pretrained on web images
!gdown --id 1fH2rq53x6JY6zE_WBKH5vP_ZGq4jvj46
dino_v3 = torch.hub.load(repo_or_dir='./dinov3-main', model='dinov3_vits16plus', source='local', weights='dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth')

# !gdown --id 1Fznrc_pDwp7iaUBhAoWUKPI1vsGpHy8m
# dino_v3 = torch.hub.load(repo_or_dir='./dinov3-main', model='dinov3_vitb16', source='local', weights='dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth')


# !gdown --id 1CfyML_a7PkVpt4Lhy-3yNxSfUgsFfgCs
# dino_v3 = torch.hub.load(repo_or_dir='./dinov3-main', model='dinov3_vitl16', source='local', weights='dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth')

dino_v3 = dino_v3.to(device)
dino_v3.eval()

print("¡Modelo DinoV3 cargado exitosamente!")
print(f"Dimensión de salida del modelo: {dino_v3.embed_dim}")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1-LqJ2bS_T8XOx5TClDq31MCkPPoXO4bZ
To: /content/dinov3-main.zip
100% 10.3M/10.3M [00:00<00:00, 304MB/s]
Archive:  dinov3-main.zip
adc254450203739c8149213a7a69d8d905b4fcfa
replace dinov3-main/.docstr.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: dinov3-main/.docstr.yaml  
  inflating: dinov3-main/.github/workflows/lint.yaml  
  inflating: dinov3-main/.gitignore  
  inflating: dinov3-main/CODE_OF_CONDUCT.md  
  inflating: dinov3-main/CONTRIBUTING.md  
  inflating: dinov3-main/LICENSE.md  
  inflating: dinov3-main/MODEL_CARD.md  
  inflating: dinov3-main/README.md   
  inflating: dinov3-main/conda.yaml  
  inflating: dinov3-main/dinov3/__init__.py  
  inflating: dinov3-main/dinov3/checkpointer/__init__.py  
 

100%|██████████| 110M/110M [00:00<00:00, 761MB/s] 


¡Modelo DinoV3 cargado exitosamente!
Dimensión de salida del modelo: 384


# Carga de imágenes

In [17]:
# Descargar el archivo zip desde Google Drive
!gdown --id 11-TD6add7zZaukIB8l2cVnBTJgsTjo8n

# Descomprimir el archivo
!unzip dataset_ecom_mini.zip

# Cargar los metadatos del conjunto de datos
data_dir = Path('eval')
df = pd.read_csv('eval.csv', delimiter=';')

print(f"El conjunto de datos contiene {len(df)} imágenes")
print(f"\nDistribución de categorías:")
print(df['GlobalCategoryEN'].value_counts())


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=11-TD6add7zZaukIB8l2cVnBTJgsTjo8n
To: /content/dataset_ecom_mini.zip
100% 10.6M/10.6M [00:00<00:00, 110MB/s]
Archive:  dataset_ecom_mini.zip
replace eval/45elec.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: eval/45elec.jpg         
  inflating: eval/im.qf53qh.input.jpg  
  inflating: eval/8toys.jpg          
  inflating: eval/im.rf7hhh.input.jpg  
  inflating: eval/im.rys7rv.input.jpg  
  inflating: eval/226pets.jpg        
  inflating: eval/0off.jpg           
  inflating: eval/im.7wd4b3.input.jpg  
  inflating: eval/443pets.jpg        
  inflating: eval/im.yy3v67.input.jpg  
  inflating: eval/im.zu5wxi.input.jpg  
  inflating: eval/39pets.jpg         
  inflating: eval/24elec.jpg         
  inflating: ev

# Paso 3:

In [21]:
def transform(image):
    transform_pipeline = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return transform_pipeline(image)

def encode_image(image_path, model):
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model(img_tensor)

    return features.cpu().numpy().flatten()

# Codificar todas las imágenes
embeddings_v2 = []
embeddings_v3 = []
valid_filenames = []

# Get all image filenames from the dataframe
image_filenames = df['Title'].values
data_dir = Path('eval') # Define data_dir here as well

for filename in tqdm(image_filenames):
    img_path = data_dir / f"{filename}.jpg"
    if img_path.exists():
        try:
            image = Image.open(img_path)
            transformed_image = transform(image)
            embeddings_v2.append(encode_image(img_path, dino_v2))
            embeddings_v3.append(encode_image(img_path, dino_v3))
            valid_filenames.append(filename)
        except Exception as e:
            print(f"Error codificando {filename}: {e}")

# Convertir a array de numpy
print(f"\nCodificadas exitosamente {len(embeddings)} imágenes")

100%|██████████| 300/300 [00:11<00:00, 26.87it/s]


Codificadas exitosamente 300 imágenes


# Paso 4

In [ ]:
# Función auxiliar para calcular similitud coseno
def cosine_similarity(vec1, vec2):
    """
    vec1, vec2: (numpy arrays)
    Puntuación de similitud entre 0 y 1 (1 = idéntico)
    """
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)

# Función auxiliar para mostrar imágenes lado a lado con sus puntuaciones de similitud.
def display_image_comparison(img_paths, titles, similarities=None):
    """
    img_paths: Lista de rutas de imágenes
    titles: Lista de títulos para cada imagen
    similarities: Lista opcional de puntuaciones de similitud
    """

    n_images = len(img_paths)
    fig, axes = plt.subplots(1, n_images, figsize=(5*n_images, 5))

    if n_images == 1:
        axes = [axes]

    for idx, (img_path, title) in enumerate(zip(img_paths, titles)):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].axis('off')

        if similarities and idx > 0:
            axes[idx].set_title(f"{title}\nSimilitud: {similarities[idx-1]:.4f}", fontsize=12)
        else:
            axes[idx].set_title(title, fontsize=12)

    plt.tight_layout()
    plt.show()

In [22]:
# Normalizar embeddings para cálculo eficiente de similitud coseno
# Similitud coseno = producto punto de vectores normalizados
embeddings_normalized_v2 = normalize(embeddings_v2, axis=1)
embeddings_normalized_v3 = normalize(embeddings_v3, axis=1)
# Calcular matriz de similitud: N x N
similarity_matrix_v2 = np.dot(embeddings_normalized_v2, embeddings_normalized_v2.T)
similarity_matrix_v3 = np.dot(embeddings_normalized_v3, embeddings_normalized_v3.T)


def print_results(similarity_matrix):
    print(f"Forma de la matriz de similitud: {similarity_matrix.shape}")
    print(f"Valores diagonales (auto-similitud) deberían ser ~1.0: {similarity_matrix.diagonal()[:5]}")
    print(f"\nEstadísticas de similitud:")
    print(f"  Mín: {similarity_matrix.min():.4f}")
    print(f"  Máx: {similarity_matrix.max():.4f}")
    print(f"  Media: {similarity_matrix.mean():.4f}")
    print(f"  Desv. Est.: {similarity_matrix.std():.4f}")

print_results(similarity_matrix_v2)
print_results(similarity_matrix_v3)

Forma de la matriz de similitud: (300, 300)
Valores diagonales (auto-similitud) deberían ser ~1.0: [1. 1. 1. 1. 1.]

Estadísticas de similitud:
  Mín: -0.1963
  Máx: 1.0000
  Media: 0.0607
  Desv. Est.: 0.1130
Forma de la matriz de similitud: (300, 300)
Valores diagonales (auto-similitud) deberían ser ~1.0: [1. 1. 1. 1. 1.]

Estadísticas de similitud:
  Mín: -0.2163
  Máx: 1.0000
  Media: 0.0662
  Desv. Est.: 0.1100
